# Titanic — Exploratory Data Analysis

**Methodology note:** the train/validation split happens in the very next cell, before any exploration. From that point on, **all analysis in this notebook uses `train_df` only** — `val_df` is set aside and left untouched. This keeps the validation set a genuinely unseen holdout: no preprocessing or feature-engineering decision made below is influenced by patterns that happen to show up in the rows that will be used for evaluation.

Goal of this notebook: understand the data well enough to justify the choices made in `src/preprocess.py` (imputation strategy, encoding, which features to engineer, which columns to drop).

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Make the `src` package importable when this notebook runs from notebooks/
sys.path.append(str(Path.cwd().parent))

from src.data import download_titanic, load_train_csv, split_train_val, TARGET

sns.set_theme(style="whitegrid")
%matplotlib inline
pd.set_option("display.max_columns", None)

In [ ]:
# Load the full train.csv, then split immediately (before any exploration).
csv_path = download_titanic()
full_df = load_train_csv(csv_path)

train_df, val_df = split_train_val(full_df)

print(f"full_df : {full_df.shape}")
print(f"train_df: {train_df.shape}  <- everything below uses this")
print(f"val_df  : {val_df.shape}    <- untouched from here on")

## 1. First look

TODO:
- `train_df.head()`, `train_df.info()`, `train_df.describe()` (numeric) and `describe(include='object')` (categorical)
- Note the dtypes: anything that *looks* numeric but is really categorical (hint: think about what a number like `Pclass` actually represents)

## 2. Missing values

TODO:
- Count + percentage of missing values per column (`isna().sum()` / `len(train_df)`)
- Sort descending so the worst offenders are obvious
- For each column with missing data, write a one-line note: is this "missing at random" or does missingness itself carry information (e.g. does *not* having a `Cabin` recorded correlate with survival)?

This directly feeds the imputation strategy in `src/preprocess.py`.

## 3. Target distribution (`Survived`)

TODO:
- `value_counts(normalize=True)` and a bar plot
- Write down the exact survival rate. This is the number a trivial "always predict the majority class" baseline would score on accuracy — it's the bar the real model has to clear, and the reason accuracy alone won't be enough in `src/evaluate.py`.

## 4. Survival by categorical features

TODO — for each of `Sex`, `Pclass`, `Embarked`:
- `groupby(col)[TARGET].mean()` (survival rate per group)
- A bar plot or `sns.barplot(x=col, y=TARGET, data=train_df)`
- One sentence per feature: how strong does the effect look, and does it match maritime-disaster intuition ("women and children first", wealthier passengers closer to lifeboats)?

## 5. `Age` and `Fare` distributions

TODO:
- Histogram (or `sns.histplot`) for each, plus `sns.boxplot` to spot outliers
- Compare `mean` vs `median` for `Fare` — a large gap is a tell for skew, and it's the justification for choosing median (not mean) imputation later
- Optional: `Age`/`Fare` split by `Survived` (e.g. `sns.histplot(..., hue=TARGET)`) to see whether the distributions actually differ by outcome

## 6. Family size (`SibSp` + `Parch`)

TODO:
- Look at `SibSp` and `Parch` individually first
- Then construct `FamilySize = SibSp + Parch + 1` (the `+1` is the passenger themself) *in a scratch column here* — just to inspect it, not to modify `train_df` in place — and check survival rate by family size
- Is the relationship monotonic, or is there a "sweet spot" (e.g. small families do better than both solo travelers and large families)? That shape is worth a sentence — it's a real, somewhat non-obvious pattern in this dataset.

## 7. Correlation matrix (numeric features)

TODO:
- `train_df.corr(numeric_only=True)` + `sns.heatmap(..., annot=True)`
- Note which numeric features correlate most with `Survived`, and whether any two features are strongly correlated with *each other* (redundant information)

## 8. Title extraction from `Name`

The `Name` field itself (e.g. `"Rugg, Miss. Emily"`) is just an identifier and shouldn't be fed to a model directly — but it encodes a social title that turns out to be informative.

TODO:
- Extract the title with a regex, e.g. `train_df['Name'].str.extract(r',\s*([^.]*)\.')`
- `value_counts()` on the extracted titles — you'll see the common ones (Mr, Miss, Mrs, Master) plus a long tail of rare ones (Dr, Rev, Col, the French/aristocratic titles, ...)
- Survival rate by title (a scratch column again, don't mutate `train_df`)
- Decide: which rare titles should be grouped into a single `"Rare"` bucket for `src/preprocess.py`? (Hint: a title that appears once or twice can't be learned from reliably, and `OneHotEncoder(handle_unknown='ignore')` will need to handle whatever shows up at inference time regardless.)

## 9. Summary — preprocessing decisions

Fill this in last, after finishing the sections above. This is the bridge to `src/preprocess.py` (Block 3) — every line here should be traceable to a specific finding above.

TODO, as a short bullet list:
- Columns to drop, and why
- Missing-value strategy per column, and why
- Engineered features to add (`Title`, `FamilySize`, ...), and why
- Which columns are numeric vs. categorical for the `ColumnTransformer`, and any surprising classification decisions (e.g. `Pclass`)